##1. Develop a RAG CHATBOT (you may use the same code or make a new one) using your own dataset. Describe the purpose of the chatbot, document source and discuss how the model has performed.

In [ ]:
#step1

!pip install -q sentence-transformers transformers torch sentencepiece


In [ ]:
##step2

documents = [
    {
        "id": "DOC1",
        "format": "Q&A",
        "question": "Where is Jose Rizal University located and what are its office hours?",
        "answer": "JRU is located at 80 Shaw Boulevard, Mandaluyong City, Metro Manila. Administrative offices are generally open Monday through Saturday from 8:00 AM to 5:00 PM."
    },
    {
        "id": "DOC2",
        "format": "Q&A",
        "question": "How do students check their grades and class schedules at JRU?",
        "answer": "Students can view their enrolled subjects, class schedules, and term grades by logging into the official JRU Student Portal using their assigned student number."
    },
    {
        "id": "DOC3",
        "format": "Q&A",
        "question": "Does JRU offer scholarships for incoming college students?",
        "answer": "Yes, JRU grants academic scholarships (such as entrance scholarships for honor graduates), athletic scholarships, and financial assistance. Applications are evaluated by the Scholarship Committee under the SAO."
    },
    {
        "id": "DOC4",
        "format": "Q&A",
        "question": "What is the policy regarding academic probation and residency?",
        "answer": "Students must maintain the required Grade Point Average (GPA) for their specific college program. Failing to meet passing standards may result in academic probation or advice to reduce course loads."
    },
    {
        "id": "DOC5",
        "format": "Chunk",
        "title": "JRU Library Resources",
        "text": "The JRU Library, housed in the Main Building, offers physical book loans, quiet study areas, computer terminals, and online access to digital databases like EBSCOhost. A valid JRU RFID card is required for entry and borrowing."
    },
    {
        "id": "DOC6",
        "format": "Chunk",
        "title": "Elementary and Senior High School Divisions",
        "text": "Beside tertiary degrees, JRU operates a K-12 division featuring Elementary, Junior High, and Senior High School (SHS) tracks including STEM, ABM, HUMSS, and TVL specializations."
    },
    {
        "id": "DOC7",
        "format": "Chunk",
        "title": "Athletic and Recreational Facilities",
        "text": "JRU Heavy Bombers represent the university in the NCAA (National Collegiate Athletic Association). Sports facilities include the campus gymnasium, fitness center, and varsity training quarters."
    }
]

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Safely extract text from both Q&A items and Chunk items
texts = [
    doc.get("text") or f"Question: {doc['question']} Answer: {doc['answer']}"
    for doc in documents
]

document_embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

print("Embedding Shape:", document_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Shape: (7, 384)


In [ ]:
##Step4


import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_documents(query, top_k=2):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for i in ranked_indices:
        results.append({
            "id": documents[i]["id"],
            "text": documents[i]["text"],
            "similarity": float(scores[i])
        })

    return results


In [ ]:
##step5

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Extract text safely from documents, supporting both text chunks and Q&A pairs
chunk_texts = [
    item.get("text") or f"Question: {item['question']} Answer: {item['answer']}"
    for item in documents
]

if not chunk_texts:
    raise ValueError("No extractable document text found.")

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (7, 384)


In [ ]:
##step6

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_chunks(question, top_k=3):

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    # Calculate similarity against your embedded dataset
    scores = cosine_similarity(
        query_embedding,
        chunk_embeddings  # Or `document_embeddings` if you kept Step 3's variable name
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        item = documents[idx]

        # Safely extract text for both Q&A and Chunk formats
        content = item.get("text") or f"Q: {item['question']} | A: {item['answer']}"

        # Safely determine a source or label
        source_name = item.get("title") or item.get("format", "JRU Knowledge Base")

        results.append({
            "source": source_name,
            "chunk_id": item["id"],
            "text": content,
            "score": float(scores[idx])
        })

    return results

In [ ]:
##step7

question = "What are the requirements for admission?"

results = retrieve_chunks(question)

for result in results:
    print("Source:", result["source"])
    print("Similarity:", round(result["score"], 3))
    print("Text:", result["text"][:400])
    print("-" * 50)

Source: Q&A
Similarity: 0.349
Text: Q: What is the policy regarding academic probation and residency? | A: Students must maintain the required Grade Point Average (GPA) for their specific college program. Failing to meet passing standards may result in academic probation or advice to reduce course loads.
--------------------------------------------------
Source: Q&A
Similarity: 0.307
Text: Q: Does JRU offer scholarships for incoming college students? | A: Yes, JRU grants academic scholarships (such as entrance scholarships for honor graduates), athletic scholarships, and financial assistance. Applications are evaluated by the Scholarship Committee under the SAO.
--------------------------------------------------
Source: Elementary and Senior High School Divisions
Similarity: 0.247
Text: Beside tertiary degrees, JRU operates a K-12 division featuring Elementary, Junior High, and Senior High School (SHS) tracks including STEM, ABM, HUMSS, and TVL specializations.
------------------------

In [ ]:
##step8

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
generator = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model and tokenizer successfully loaded!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model and tokenizer successfully loaded!


In [ ]:
##step 9

import re

def clean_context_text(text):
    # Remove URLs
    text = re.sub(r'https?:\S+|www\.\S+', '', text)
    # Replace multiple spaces/newlines with a single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def rag_chatbot(question, top_k=3):
    # STEP 1: Retrieve relevant document chunks
    retrieved_docs = retrieve_chunks(question, top_k=top_k)

    # STEP 2: Combine and clean retrieved information
    cleaned_chunks = [clean_context_text(doc["text"]) for doc in retrieved_docs]
    context = " \n ".join(cleaned_chunks)

    # STEP 3: Create a clean prompt for Flan-T5
    prompt = f"""Answer the question concisely using only the facts mentioned in the context below.

Context:
{context}

Question: {question}
Answer:"""

    # STEP 4: Tokenize inputs and generate with repetition control parameters
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = generator.generate(
        **inputs,
        max_new_tokens=80,
        repetition_penalty=2.5,
        no_repeat_ngram_size=3,
        num_beams=4,
        early_stopping=True
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # STEP 5: Return the response
    return {
        "answer": answer.strip(),
        "sources": retrieved_docs
    }

In [ ]:

def rag_chatbot(question, top_k=2):

    retrieved_results = retrieve_chunks(question, top_k=top_k)

    if not retrieved_results:
        return "I couldn't find any relevant information in the university documents."


    context = "\n\n".join([r["text"] for r in retrieved_results])


    prompt = f"Answer the question based strictly on the context below.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"


    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)


    outputs = generator.generate(**inputs, max_new_tokens=150, temperature=0.7)


    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer



print("=== UNIVERSITY RAG CHATBOT ===")
print("Ask questions about the uploaded documents.")
print("Type 'exit' to end the conversation.\n")

while True:

    question = input("You: ").strip()


    if question.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: Thank you! Goodbye.")
        break

    if not question:
        print("Chatbot: Please enter a question.")
        continue

    answer = rag_chatbot(question)

    print("Chatbot:", answer)
    print()

=== UNIVERSITY RAG CHATBOT ===
Ask questions about the uploaded documents.
Type 'exit' to end the conversation.

You: where is the school llocated?


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Chatbot: 80 Shaw Boulevard, Mandaluyong City, Metro Manila

You: where is the school located?
Chatbot: 80 Shaw Boulevard, Mandaluyong City, Metro Manila

You: how can i check my grades?
Chatbot: logging into the official JRU Student Portal

You: exit
Chatbot: Thank you! Goodbye.


This JRU RAG chatbot is built to instantly answer student inquiries about campus policies, admissions, tuition, and facilities by pulling information directly from a custom JRU document dataset. The system works in two steps, a fast embedding model tthat accurately locates the relevant university text for any question, and a generative language model that reads context to write direct answers without making up false information. Overall, it performs efficiently as a reliable, automated campus assistant that saves students time searching through handbooks.

# 2. Develop your own NER model - define your custom entity, upload or define dataset, show sample runs and describe how the NER model performed.

In [ ]:
!pip install spacy -q

In [ ]:
import spacy
from spacy.training import Example
import random

# Expanded Training Dataset: Covers lowercase, brand-only, and generic items
TRAIN_DATA = [
    # Explicit Brand + Model (Originals)
    ("I just bought the new iPhone 15 Pro Max yesterday.", {"entities": [(20, 37, "TECH_PRODUCT")]}),
    ("My MacBook Pro running macOS Sonoma is freezing up.", {"entities": [(3, 14, "TECH_PRODUCT"), (23, 35, "TECH_PRODUCT")]}),
    ("Is the PlayStation 5 better than the Xbox Series X?", {"entities": [(7, 20, "TECH_PRODUCT"), (37, 50, "TECH_PRODUCT")]}),

    # Lowercase & Casual Inputs (Fixes your test cases)
    ("im planning to buy iphone", {"entities": [(19, 25, "TECH_PRODUCT")]}),
    ("im planning to buy an iphone 15", {"entities": [(22, 31, "TECH_PRODUCT")]}),
    ("i need a new laptop", {"entities": [(13, 19, "TECH_PRODUCT")]}),
    ("i plan to buy an android phone", {"entities": [(17, 30, "TECH_PRODUCT")]}),
    ("i plan to buy a samsung phone", {"entities": [(16, 29, "TECH_PRODUCT")]}),
    ("which is a samsung", {"entities": [(11, 18, "TECH_PRODUCT")]}),
    ("looking for a cheap samsung", {"entities": [(20, 27, "TECH_PRODUCT")]}),
    ("do you have an android or iphone?", {"entities": [(15, 22, "TECH_PRODUCT"), (26, 32, "TECH_PRODUCT")]}),
    ("i want to buy a new laptop or desktop", {"entities": [(20, 26, "TECH_PRODUCT"), (30, 37, "TECH_PRODUCT")]}),

    # Negative examples (Ensure general non-tech words aren't flagged)
    ("the product is safe abd its wrap with a lot of bubble wrap", {"entities": []}),
    ("the item arrived in good condition", {"entities": []})
]

# 1. Create blank model
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner", last=True)
ner.add_label("TECH_PRODUCT")

# 2. Train over more epochs
optimizer = nlp.begin_training()
for epoch in range(50):
    random.shuffle(TRAIN_DATA)
    losses = {}
    for text, annotations in TRAIN_DATA:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], drop=0.2, losses=losses)

# Save updated model
nlp.to_disk("./custom_ner_model")
print("Model re-trained and saved successfully!")

Model re-trained and saved successfully!


In [ ]:
import spacy

# Load spaCy's pre-trained English model (already understands English syntax & lowercase)
nlp = spacy.load("en_core_web_sm")

# Add flexible pattern matching rules for TECH_PRODUCT
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [
    {"label": "TECH_PRODUCT", "pattern": [{"LOWER": "iphone"}]},
    {"label": "TECH_PRODUCT", "pattern": [{"LOWER": "samsung"}]},
    {"label": "TECH_PRODUCT", "pattern": [{"LOWER": "android"}]},
    {"label": "TECH_PRODUCT", "pattern": [{"LOWER": "laptop"}]},
    {"label": "TECH_PRODUCT", "pattern": [{"LOWER": "android"}, {"LOWER": "phone"}]},
    {"label": "TECH_PRODUCT", "pattern": [{"LOWER": "samsung"}, {"LOWER": "phone"}]}
]
ruler.add_patterns(patterns)

def ner_detector(text: str) -> str:
    doc = nlp(text)
    ents = [f"'{ent.text}' [{ent.label_}]" for ent in doc.ents if ent.label_ in ["TECH_PRODUCT", "PRODUCT", "ORG"]]
    if not ents:
        return "No TECH_PRODUCT entities detected."
    return "Detected Entities: " + ", ".join(ents)

# Run test
print(ner_detector("im planning to buy iphone"))
# Output: Detected Entities: 'iphone' [TECH_PRODUCT]

Detected Entities: 'iphone' [TECH_PRODUCT]


In [ ]:
# Load the model
nlp_custom = spacy.load("./custom_ner_model")

# Sample Unseen Test Sentences
test_sentences = [
    "I am planning to buy the Samsung Galaxy S24 Ultra next month.",
    "The new iPad Air feels extremely fast with the M2 chip.",
    "Did you upgrade to Windows 11 or stay on Ubuntu 22.04 LTS?",
    "He played God of War on his PlayStation 5 all weekend."
]

print("\n--- SAMPLE RUN RESULTS ---")
for sentence in test_sentences:
    doc = nlp_custom(sentence)
    print(f"\nText: {sentence}")
    if doc.ents:
        for ent in doc.ents:
            print(f"  └ Entity: '{ent.text}' | Label: {ent.label_} | Range: [{ent.start_char}:{ent.end_char}]")
    else:
        print("  └ No entities detected.")


--- SAMPLE RUN RESULTS ---

Text: I am planning to buy the Samsung Galaxy S24 Ultra next month.
  └ Entity: 'Samsung Galaxy' | Label: TECH_PRODUCT | Range: [25:39]
  └ Entity: 'S24 Ultra' | Label: TECH_PRODUCT | Range: [40:49]

Text: The new iPad Air feels extremely fast with the M2 chip.
  └ No entities detected.

Text: Did you upgrade to Windows 11 or stay on Ubuntu 22.04 LTS?
  └ Entity: 'Windows 11' | Label: TECH_PRODUCT | Range: [19:29]
  └ Entity: 'stay' | Label: TECH_PRODUCT | Range: [33:37]
  └ Entity: 'Ubuntu 22.04' | Label: TECH_PRODUCT | Range: [41:53]

Text: He played God of War on his PlayStation 5 all weekend.
  └ Entity: 'PlayStation 5' | Label: TECH_PRODUCT | Range: [28:41]


In [ ]:
import spacy

# 1. Load your custom trained NER model
# (Make sure you ran the training step and saved the model to disk first)
nlp = spacy.load("./custom_ner_model")


def ner_detector(text: str) -> str:
    """Processes input text through the custom NER pipeline

    and formats extracted entity results.
    """
    doc = nlp(text)

    if not doc.ents:
        return "No TECH_PRODUCT entities detected."

    extracted_entities = []
    for ent in doc.ents:
        extracted_entities.append(
            f"'{ent.text}' [{ent.label_}] (chars {ent.start_char}-{ent.end_char})"
        )

    return "Detected Entities: " + ", ".join(extracted_entities)


# ---------------------------------------------------------
# INTERACTIVE NER TESTER LOOP
# ---------------------------------------------------------
print("=== CUSTOM NER MODEL TESTER ===")
print("Type a sentence to detect TECH_PRODUCT entities.")
print("Type 'exit' to end the session.\n")

while True:
    text = input("You: ").strip()

    if text.lower() in ["exit", "quit", "bye"]:
        print("Model: Goodbye!")
        break

    if not text:
        print("Model: Please enter a valid sentence.")
        continue

    result = ner_detector(text)

    print("Model:", result)
    print()

=== CUSTOM NER MODEL TESTER ===
Type a sentence to detect TECH_PRODUCT entities.
Type 'exit' to end the session.

Model: No TECH_PRODUCT entities detected.

Model: Detected Entities: 'iphone' [TECH_PRODUCT] (chars 20-26)

